# rPPG · 三步跑完

| 格 | 干什么 | 什么时候要改 |
|---|---|---|
| 第 0 格 | 环境：挂 Drive → 拉代码（钉死版本）→ 判新 VM → 装依赖/补丁 → 凭据 → 数据视图 | **永不改**；换机器/换硬件时重跑一次即可 |
| 第 1 格 | 跑实验：`run("名字", ctx)` | 只改实验名和参数 |
| 第 2 格 | 汇总：所有日志里的 FFT 指标 + 版本戳 | 不用改 |

所有实验都定义在 `projects/rppg/experiments.py` 里，不在 notebook 里。
原因：CPU 调试和 GPU 复跑是**两台不同的 VM**，内核变量不会跟着走 ——
"调通的成果"必须落成进 git 的代码 + 进 Drive 的日志，才能复用。

> **只转圈、没有任何输出**：内核被上一轮卡死了（上一版 `drive.mount()` 会静默干等）。先 `Restart Kernel`，再重跑本格。新版带看门狗，最多等 90 秒就放行报错，不会再占死内核。


In [ ]:
# ============ 第 0 格：环境（固定块，永远不要改）============
# 幂等：挂 Drive → 拉代码 → 判新 VM → 装依赖/打补丁 → 载凭据 → 备数据
# 如果这一格只转圈、一个字都不打印：内核被上一轮卡死了 → 先 Restart Kernel
print("[0/3] 同步代码仓库…", flush=True)

import os, subprocess, sys
from pathlib import Path

os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")   # git 绝不弹账号密码，宁可报错
repo = Path("/content/colab-workbench")
try:
    if not repo.exists():
        subprocess.run(["git", "clone", "-q",
            "https://github.com/Elisabethhui/colab-workbench.git",
            str(repo)], check=True, timeout=180)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "-q", "--ff-only"],
                       check=False, timeout=180)
except subprocess.TimeoutExpired:
    raise RuntimeError("git 同步超过 180 秒，网络可能不通。先确认 VM 能访问 github.com") from None
head = subprocess.run(["git", "-C", str(repo), "log", "--oneline", "-1"],
                      capture_output=True, text=True).stdout.strip()
print("[1/3] 代码版本:", head, flush=True)

# 同一个内核里重跑时，必须丢掉旧的模块缓存，否则改动不生效
for _m in [m for m in sys.modules if m == "bootstrap" or m.startswith("projects")]:
    del sys.modules[_m]
sys.path.insert(0, str(repo))

print("[2/3] 进入 bootstrap.setup()（下一步是挂 Drive，会弹授权）", flush=True)
from bootstrap import setup
ctx = setup("rppg")
print("[3/3] 完成", flush=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第 1 格 · 跑实验

实验名固定，一个名字 = 一份配置 + 一段日志：

| 名字 | 硬件 | 预期 | 说明 |
|---|---|---|---|
| `baseline_classical` | CPU | 8–20 分钟 | POS/CHROM 古典基线，不需要权重，**CPU 就能出真指标** |
| `effphys_pure` | GPU | ~6 分钟 | PURE 训的 EfficientPhys → UBFC 全部 42 个 |
| `effphys_scamps` | GPU | ~6 分钟 | SCAMPS 训的 EfficientPhys → UBFC 全部 42 个 |
| `train_e1` | GPU | 20–40 分钟 | 自己训练：UBFC 前 80% 训 / 后 20% 测（30 epoch） |
| `author_test20` | GPU | ~2 分钟 | 作者预训练模型只跑后 20%，和 E1 同区间对照 |

三个开关：

* `limit=2` —— 只跑前 2 个 subject（冒烟，1~2 分钟）
* `dry_run=True` —— 只生成配置并打印关键字段，不执行（CPU 阶段验证用）
* `force=True` —— 硬件不够也硬跑（GPU 实验在 CPU 上会非常慢）

当前内核没有 GPU 时，GPU 实验会**直接跳过并提示**，不会白跑 40 分钟。


In [ ]:
import sys
from pathlib import Path
import bootstrap

# experiments.py 在工作流仓库里（ctx.paths.repo 是上游 rPPG-Toolbox，别弄混）
sys.path.insert(0, str(Path(bootstrap.WB_HOME) / "projects" / "rppg"))
from experiments import run, summary, list_experiments

list_experiments()                      # 先看有什么、各要什么硬件

# run("baseline_classical", ctx)          # ← 改这里。冒烟就加 limit=2，验配置就加 dry_run=True
run("effphys_pure", ctx)            


## 第 2 格 · 汇总

把所有日志（在 Drive 上的 `logs/`，换机器不丢）里的 FFT 指标拉成一张表，
并带上版本戳：哪个上游 commit + 什么硬件跑出来的。换机器后重跑第 0 格再来这格，就能看到历史。


In [ ]:
summary(ctx)

## 第 3 格 · 流式验证（真摄像头 / 录像回放）

离线口径是「整段视频 → 一个心率」；流式是「每来一帧进缓冲，每 0.5 秒用过去 6 秒算一次当前心率」。

- `SOURCE = 0` 表示读摄像头。**Colab VM 没有摄像头**，会自动回退到 Drive 上的 `inputs/cam_clip.mp4`
- 想验真·实时延迟：`SOURCE = 0` 在本机 Mac 跑（`python stream_demo.py --source 0 --device cpu`）
- 这格同时记录**单窗推理耗时**和**端到端延迟**（窗口时长 + 推理），这是流式方案的真实代价


In [ ]:
# ============ 第 3 格：流式验证 ============
# 前提：第 0 格跑过（ctx 存在、Drive 已挂、repo 已钉死版本）
import sys, time
from pathlib import Path
import numpy as np, cv2, torch

# ---------------- 可调 ----------------
SOURCE = 0          # 0 = 摄像头；或填视频路径
WINDOW, STRIDE = 180, 15      # 6 秒窗 / 每 0.5 秒出一次（30fps 下）
CLIP = Path("/content/drive/MyDrive/Colab_Projects/projects/rppg/inputs/cam_clip.mp4")

PROJECT = Path(bootstrap.WB_HOME) / "projects" / "rppg"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
sys.modules.pop("stream_demo", None)      # 改过 stream_demo.py 后能立刻生效
import stream_demo as sd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = sd.load_model(str(ctx.paths.repo),
                      str(Path(ctx.paths.repo) / "final_model_release" / "PURE_EfficientPhys.pth"),
                      device)
cropper = sd.FaceCropper(size=72, detect_every=30)


def open_source(source):
    """摄像头优先，打不开就回退录像（Colab VM 必然回退）。"""
    if str(source) == "0":
        cap = cv2.VideoCapture(0)
        if cap.isOpened():
            ok, _ = cap.read()
            if ok:
                cap.release()
                return cv2.VideoCapture(0), "摄像头"
            cap.release()
        print("[信息] 摄像头打不开（Colab VM 无摄像头）→ 回退录像")
    for cand in (source, CLIP):
        if str(cand) != "0" and Path(cand).exists():
            return cv2.VideoCapture(str(cand)), f"录像 {Path(cand).name}"
    raise FileNotFoundError("既没有摄像头，也没有 " + str(CLIP))


cap, src_name = open_source(SOURCE)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
print(f"[流式] {src_name}  {fps:.0f} fps  窗口 {WINDOW} 帧({WINDOW/fps:.1f}s)  "
      f"刷新 {STRIDE} 帧({STRIDE/fps:.2f}s)  设备 {device}")

frames, rows, hr_hist = [], [], []
idx = n_infer = fail = 0
t_start = time.time()
while True:
    ok, frame = cap.read()
    if not ok:
        fail += 1
        if fail >= (30 if str(SOURCE) == "0" else 1):
            break
        time.sleep(0.05)
        continue
    fail = 0
    frames.append(cropper.crop(frame, idx))
    if len(frames) > WINDOW + 1:
        frames.pop(0)
    idx += 1
    if len(frames) < WINDOW + 1 or (idx - WINDOW) % STRIDE:
        continue

    t0 = time.perf_counter()
    wave = sd.forward_window(model, np.stack(frames), device)
    ms = (time.perf_counter() - t0) * 1000
    hr = sd.hr_from_waveform(wave, fps, 0.75, 2.5)
    n_infer += 1
    if np.isfinite(hr):
        hr_hist.append(hr)
        hr_s = float(np.median(hr_hist[-5:]))
    else:
        hr_s = float("nan")
    rows.append({"t": idx / fps, "hr": hr_s, "ms": ms})
    print(f"t={idx/fps:6.1f}s  HR={hr_s:5.1f} bpm (本窗 {hr:5.1f})  "
          f"推理 {ms:5.1f} ms  端到端延迟 {(WINDOW/fps)+ms/1000:4.1f}s")

cap.release()
elapsed = time.time() - t_start
if not rows:
    print("[结果] 帧数不够一个窗口，没有输出")
else:
    ms_all = np.array([r["ms"] for r in rows])
    hr_all = np.array([r["hr"] for r in rows if np.isfinite(r["hr"])])
    print(f"\n[汇总] {idx} 帧 / {elapsed:.1f}s（{idx/elapsed:.1f} fps 有效）  "
          f"推理 {n_infer} 次  中位 {np.median(ms_all):.1f} ms  p95 {np.percentile(ms_all,95):.1f} ms")
    if len(hr_all):
        print(f"[心率] 中位 {np.median(hr_all):.1f} bpm  "
              f"范围 {hr_all.min():.1f}-{hr_all.max():.1f}  抖动(std) {hr_all.std():.1f}")
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(11, 3))
        ax[0].plot([r["t"] for r in rows], [r["hr"] for r in rows], "o-")
        ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("HR (bpm)"); ax[0].set_title("rolling HR")
        ax[1].plot([r["t"] for r in rows], [r["ms"] for r in rows], ".-")
        ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("ms"); ax[1].set_title(f"{device} per-window latency")
        plt.tight_layout(); plt.show()
    except Exception as e:
        print("[绘图跳过]", e)
